In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU is connected")

CUDA available: True
CUDA version: 12.8
GPU: Tesla T4


In [3]:
#!pip uninstall -y transformers trl peft accelerate bitsandbytes

In [4]:
!pip install -q \
transformers==4.51.3 \
trl==0.17.0 \
peft==0.15.2 \
accelerate==1.6.0 \
bitsandbytes==0.46.0 \
datasets==3.5.0

In [5]:

import transformers, trl, peft, accelerate, bitsandbytes

print(transformers.__version__)
print(trl.__version__)
print(peft.__version__)
print(accelerate.__version__)
print(bitsandbytes.__version__)

4.51.3
0.17.0
0.15.2
1.6.0
0.46.0


In [6]:
#!pip install -q transformers datasets peft trl bitsandbytes accelerate wandb

In [7]:
#import wandb

#from google.colab import userdata

#wandb.login(key=userdata.get("WANDB_API_KEY"))

In [8]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

from trl import (
    SFTTrainer,
    SFTConfig,
)

In [9]:
# Hyperparameters

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

MAX_SEQUENCE_LENGTH = 1024

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]

OUTPUT_DIR = "ai-learning-coach-qwen-lora"

In [10]:
from google.colab import files

uploaded = files.upload()

Saving sft_dataset.jsonl to sft_dataset.jsonl


In [11]:
# Load JSONL
dataset = load_dataset(
    "json",
    data_files="sft_dataset.jsonl",
    split="train"
)

# First split: 80% train, 20% temp
split = dataset.train_test_split(
    test_size=0.2,
    seed=42,
)

train_dataset = split["train"]
temp_dataset = split["test"]

# Second split: split temp equally into val and test
split = temp_dataset.train_test_split(
    test_size=0.5,
    seed=42,
)

val_dataset = split["train"]
test_dataset = split["test"]

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Train: 492
Validation: 62
Test: 62


In [12]:
# Load the Qwen tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
def to_prompt_completion(example):
    return {
        "prompt": [example["messages"][0]],
        "completion": [example["messages"][1]],
    }

train_dataset = train_dataset.map(
    to_prompt_completion,
    remove_columns=["messages"],
)

val_dataset = val_dataset.map(
    to_prompt_completion,
    remove_columns=["messages"],
)

test_dataset = test_dataset.map(
    to_prompt_completion,
    remove_columns=["messages"],
)

Map:   0%|          | 0/492 [00:00<?, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

In [14]:
# convert messages into Qwen chat text
#def format_example(example):
 #   text = tokenizer.apply_chat_template(
  #      example["messages"],
   #     tokenize=False,
   #     add_generation_prompt=False,
    #)

    #return {"text": text}

In [15]:
train_dataset[0]

{'prompt': [{'content': 'What is the difference between a base model and a chat model?',
   'role': 'user'}],
 'completion': [{'content': 'What is it?  \nA base model is a type of AI model that has been trained on a large amount of text data. It learns patterns in language and can generate coherent text, but it does not have a specific focus or conversational structure. A chat model, on the other hand, is built on top of a base model and is specifically fine-tuned for conversations. It understands context, maintains dialogue flow, and can provide responses that feel natural in a chat setting.\n\nWhy do we need it?  \nBase models serve as the foundation for various natural language tasks. They provide the ability to generate text and understand language. However, for applications like chatbots or virtual assistants, we need chat models. These chat models are tailored to handle user interactions, providing relevant responses while considering the conversational context. This focus enhanc

In [16]:
#train_dataset = train_dataset.map(format_example)
#val_dataset = val_dataset.map(format_example)
#test_dataset = test_dataset.map(format_example)

In [17]:
# Load Qwen in 4-bit
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
)

model.config.use_cache = False

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [18]:
# Prepare the 4-bit model and add LoRA

from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)




In [19]:
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

In [20]:
bf16_params = [
    name
    for name, param in model.named_parameters()
    if param.requires_grad and param.dtype == torch.bfloat16
]

print(len(bf16_params))

0


In [21]:
model.print_trainable_parameters()

trainable params: 14,745,600 || all params: 3,100,684,288 || trainable%: 0.4756


In [22]:
# create sft config
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    max_length=MAX_SEQUENCE_LENGTH,
    fp16=True,
    bf16=False,
    report_to="none",

)

In [23]:
# create SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

Converting train dataset to ChatML:   0%|          | 0/492 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/492 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/492 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/492 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/62 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [24]:
print(training_args.fp16)
print(training_args.bf16)

print(model.dtype)

import torch
print(torch.cuda.get_device_name(0))

True
False
torch.float32
Tesla T4


In [25]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
10,1.113300,1.066745
20,1.036800,1.015814
30,0.996900,0.998803
40,0.932600,0.992888
50,0.966800,0.987688
60,0.962700,0.984909
70,0.944300,0.984156
80,0.901100,0.983090
90,0.937400,0.982600


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=90, training_loss=1.01239013671875, metrics={'train_runtime': 1670.627, 'train_samples_per_second': 0.884, 'train_steps_per_second': 0.054, 'total_flos': 1.1936074389504e+16, 'train_loss': 1.01239013671875})

In [26]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('ai-learning-coach-qwen-lora/tokenizer_config.json',
 'ai-learning-coach-qwen-lora/special_tokens_map.json',
 'ai-learning-coach-qwen-lora/vocab.json',
 'ai-learning-coach-qwen-lora/merges.txt',
 'ai-learning-coach-qwen-lora/added_tokens.json',
 'ai-learning-coach-qwen-lora/tokenizer.json')

In [27]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [28]:
def ask(question):
    messages = [
        {
            "role": "user",
            "content": question,
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    print(response)

In [29]:
ask("Explain Random Forest.")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


What is it?  
Random Forest is an ensemble learning method used for both classification and regression tasks. It works by creating multiple decision trees during the training phase and then combining their predictions to make a final decision. Each tree in the forest is trained on a random subset of the data and features, which helps reduce overfitting.

Why do we need it?  
We need Random Forest because it can handle large datasets with many variables effectively. Traditional decision trees can be prone to overfitting, especially when dealing with noisy or complex data. By averaging the predictions from multiple trees, Random Forest reduces this risk and improves model accuracy.

Real-world analogy  
Think of Random Forest like a group of experts making decisions about a project. Instead of relying on just one expert's opinion, they gather input from several different individuals. Each expert may have a unique perspective, and by considering multiple viewpoints, they can make a more i